In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [3]:
!pip install torchvision
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10


   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.4 MB ? eta -:--:--
   ---------------------- ----------------- 0.8/1.4 MB 2.5 MB/s eta 0:00:01
   -------------------------------------- - 1.3/1.4 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 2.1 MB/s  0:00:00


In [37]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
    
])

trainset=CIFAR10(root="./data",train=True,download=False,transform=transform)
testset=CIFAR10(root="./data",train=False,download=False,transform=transform)


In [38]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [39]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=32)

In [40]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.convo_layers=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), #kernal size=2 and stride=2

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self,x):
        x=self.convo_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x
        
     
    

In [41]:
model=CNN()

In [42]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

training the cnn

In [43]:
epochs=10

for epoch in range(epochs):
    epoch_training_loss=0.0

    for images,labels in trainloader:
        optimizer.zero_grad()
        output=model.forward(images)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss+=loss.item()

    print(f"epoch={epoch} & loss={epoch_training_loss/len(trainloader)}")
    
    

epoch=0 & loss=1.363693446988035


epoch=1 & loss=0.9415497875122159


epoch=2 & loss=0.7538605443275798


epoch=3 & loss=0.6258742223729563


epoch=4 & loss=0.5170948604488617


epoch=5 & loss=0.42839652402779027


epoch=6 & loss=0.344815529949601


epoch=7 & loss=0.2698049352354253


epoch=8 & loss=0.21473995112644895


epoch=9 & loss=0.1622442425445408


In [44]:
#evalute 
correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs=model.forward(images)
        _,predicated=torch.max(outputs,1)

        correct_labels+=(predicated==labels).sum().item()
        total_labels+=labels.size(0)
print(f"accuracy={correct_labels/total_labels*100}")

accuracy=74.5
